In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/FYP

Mounted at /content/drive
/content/drive/MyDrive/FYP


In [ ]:
import pandas as pd
data = pd.read_csv('trawlers.csv')

In [ ]:
print(data.shape[0])

4369101


In [ ]:
import pandas as pd
from geopy.distance import geodesic

data = data.sort_values(by=['mmsi', 'timestamp'])
grouped = data.groupby('mmsi')

In [ ]:
data_sorted['distance_from_shore']=data['distance_from_shore']
data_sorted['distance_from_port']=data['distance_from_port']
data_sorted['course']=data['course']

In [ ]:
print(data_sorted.head())

In [ ]:
import pandas as pd
from geopy.distance import geodesic

data_sorted = data.sort_values(by=['mmsi', 'timestamp'])

distances = []
for mmsi, group in data_sorted.groupby('mmsi'):
    latitudes = group['lat'].tolist()
    longitudes = group['lon'].tolist()
    dist_list = [geodesic((latitudes[i], longitudes[i]), (latitudes[i + 1], longitudes[i + 1])).meters for i in range(len(latitudes) - 1)]
    dist_list.append(0)
    distances.extend(dist_list)

data_sorted['distances'] = distances

print(data_sorted.head())

           mmsi     timestamp  distance_from_shore  distance_from_port  speed  \
0  9.924005e+12  1.379601e+09                  0.0         1414.178833    0.0   
1  9.924005e+12  1.379602e+09                  0.0         1414.178833    0.0   
2  9.924005e+12  1.379604e+09                  0.0         1414.178833    0.1   
3  9.924005e+12  1.379605e+09                  0.0         1414.178833    0.1   
4  9.924005e+12  1.379608e+09                  0.0         1414.178833    0.0   

       course       lat        lon  is_fishing           source  distances  
0  298.500000  8.861500 -79.668427        -1.0  false_positives   1.833676  
1  298.500000  8.861506 -79.668442        -1.0  false_positives   5.062912  
2  128.399994  8.861511 -79.668488        -1.0  false_positives   0.839230  
3  111.199997  8.861511 -79.668480        -1.0  false_positives   2.729710  
4   41.700001  8.861502 -79.668503        -1.0  false_positives   2.623719  


In [ ]:
def calculate_heading(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians(lat1), np.radians(lon1), np.radians(lat2), np.radians(lon2)
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    heading = np.degrees(np.arctan2(y, x))
    heading = (heading + 360) % 360  #normalize to [0, 360)
    return heading

lat_shifted = data_sorted['lat'].shift(-1)
lon_shifted = data_sorted['lon'].shift(-1)

data_sorted['headings'] = calculate_heading(data_sorted['lat'], data_sorted['lon'], lat_shifted, lon_shifted)

print(df)

In [ ]:
data_sorted = data_sorted.drop(columns=['source'])

In [ ]:
# Check number of NaN values before applying AR
nan_count_before = data_sorted.isnull().sum()
print(f"Number of NaN values before AR:\n{nan_count_before}")

Number of NaN values before AR:
mmsi                    0
timestamp               0
speed                  78
lat                     0
lon                     0
is_fishing              0
distances               0
headings                1
distance_from_shore     0
distance_from_port      0
course                 78
dtype: int64


In [ ]:
clean_data = data_sorted.dropna()

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
import pandas as pd

exclude_cols = ['mmsi', 'timestamp']

for col in data_sorted.columns:
    if col not in exclude_cols:
        model = AutoReg(data_sorted[col].dropna(), lags=1)  #lag of 1 for simplicity
        model_fit = model.fit()
        nan_indices = data_sorted[col].isnull()
        nan_count = nan_indices.sum()
        if nan_count > 0:
            forecast = model_fit.predict(start=len(data_sorted) + nan_count - 1, end=len(data_sorted) + 2*nan_count - 2)
            data_sorted.loc[nan_indices, col] = forecast[-nan_count:]
            print(f"Filled {nan_count} missing values in column {col} using AR model")


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/deterministic.py:307: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_inde

Filled 78 missing values in column speed using AR model
Filled 1 missing values in column headings using AR model


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Filled 78 missing values in column course using AR model


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/deterministic.py:307: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)


In [ ]:
# Check number of NaN values after applying AR
nan_count_after = clean_data.isnull().sum()
print(f"Number of NaN values after AR:\n{nan_count_after}")

Number of NaN values after AR:
mmsi                   0
timestamp              0
speed                  0
lat                    0
lon                    0
is_fishing             0
distances              0
headings               0
distance_from_shore    0
distance_from_port     0
course                 0
dtype: int64


In [ ]:
# Pre processing function
def preprocess_data(data_sorted):
    label_encoder = LabelEncoder()
    features = []
    labels = []
    for index, row in data_sorted.iterrows():
        features.append({
            'mmsi': row['mmsi'],
            'timestamp':row['timestamp'],
            'lat': row['lat'],
            'lon': row['lon'],
            'course': row['course'],
            'distance_from_shore': row['distance_from_shore'],
            'distance_from_port': row['distance_from_port'],
            'speed': row['speed'],
            'distances': row['distances'],
            'headings': row['headings'],
        })
        labels.append(1 if row['is_fishing'] > 0 else 0)

    features_df = pd.DataFrame(features)

    return features_df, np.array(labels)

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

preprocessed_data = []
for gear_type, group_data in clean_data.groupby('mmsi'):
    features, labels = preprocess_data(group_data)  # Call your existing preprocess_data function
    preprocessed_data.extend([(features_row, labels_row) for features_row, labels_row in zip(features.itertuples(index=False), labels)])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
features_df = pd.DataFrame([data_point[0] for data_point in preprocessed_data])
labels_df = pd.DataFrame([data_point[1] for data_point in preprocessed_data])

from keras.models import Sequential
from keras.layers import Dense

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(features_df, labels_df, test_size=0.2, random_state=42)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

#model architecture definition
model = Sequential()
#conv1D layer with 64 filters and a kernel size of 3
model.add(Conv1D(64, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
#maxPooling1D layer
model.add(MaxPooling1D(2))
#flatten layer to flatten the output of the convolutional layer
model.add(Flatten())
model.add(Dense(2, activation='softmax'))

from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.005)

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
87379/87379 [==============================] - 219s 2ms/step - loss: 3073224448.0000 - accuracy: 0.9718 - val_loss: 9449.0596 - val_accuracy: 0.9848
Epoch 2/10
87379/87379 [==============================] - 214s 2ms/step - loss: 13813190.0000 - accuracy: 0.9732 - val_loss: 1042.6559 - val_accuracy: 0.9451
Epoch 3/10
87379/87379 [==============================] - 226s 3ms/step - loss: 8904260.0000 - accuracy: 0.9727 - val_loss: 1675.0657 - val_accuracy: 0.9859
Epoch 4/10
87379/87379 [==============================] - 212s 2ms/step - loss: 5032273.0000 - accuracy: 0.9730 - val_loss: 438.5111 - val_accuracy: 0.9840
Epoch 5/10
87379/87379 [==============================] - 210s 2ms/step - loss: 2437826.2500 - accuracy: 0.9761 - val_loss: 1.7095 - val_accuracy: 0.9797
Epoch 6/10
87379/87379 [==============================] - 222s 3ms/step - loss: 2887338.5000 - accuracy: 0.9816 - val_loss: 0.7169 - val_accuracy: 0.9847
Epoch 7/10
87379/87379 [==============================] - 222

In [ ]:
# Evaluate the model on the testing data
accuracy = model.evaluate(X_val, y_val)

print("Accuracy:", accuracy)

27306/27306 [==============================] - 45s 2ms/step - loss: 0.1844 - accuracy: 0.9838
Accuracy: [0.18435826897621155, 0.9837878942489624]


In [ ]:
predictions = model.predict(X_val)


27306/27306 [==============================] - 49s 2ms/step


In [ ]:
predictions

array([[9.9999756e-01, 2.3383614e-06],
       [9.9999994e-01, 0.0000000e+00],
       [9.8432171e-01, 1.5678331e-02],
       ...,
       [9.8432171e-01, 1.5678331e-02],
       [9.9779886e-01, 2.2010668e-03],
       [9.8432171e-01, 1.5678331e-02]], dtype=float32)

In [ ]:
import numpy as np

# Assuming predictions is your array of predictions
threshold = 0.5
binary_predictions = np.where(predictions[:, 1] >= threshold, 1, 0)

# Print number of 1s and 0s
num_1s = np.sum(binary_predictions)
num_0s = len(binary_predictions) - num_1s
print("Number of 1s predicted:", num_1s)
print("Number of 0s predicted:", num_0s)

Number of 1s predicted: 4067
Number of 0s predicted: 869723


In [ ]:
# Get the indices of the rows where predictions indicate fishing (1)
fishing_indices = np.where(binary_predictions == 1)[0]

# Get the mmsi values corresponding to the fishing predictions
mmsi_fishing = X_val.iloc[fishing_indices]['mmsi'].unique()

# Slice data_sorted for the mmsi predicted as 1
sliced_data = data_sorted[data_sorted['mmsi'].isin(mmsi_fishing)]

In [ ]:
sliced_data

,mmsi,timestamp,speed,lat,lon,is_fishing,distances,headings,distance_from_shore,distance_from_port,course
0,1.252340e+12,1.325376e+09,0.0,52.458649,4.581200,-1.0,3.106843,313.161342,0.0,0.000000,153.000000
1,1.252340e+12,1.325378e+09,0.0,52.458668,4.581167,-1.0,3.985227,163.500463,0.0,0.000000,153.000000
2,1.252340e+12,1.325379e+09,0.0,52.458633,4.581183,-1.0,3.803353,63.427922,0.0,0.000000,153.000000
3,1.252340e+12,1.325380e+09,0.0,52.458649,4.581234,-1.0,3.403312,270.000020,0.0,0.000000,153.000000
4,1.252340e+12,1.325381e+09,0.0,52.458649,4.581183,-1.0,5.966527,22.299895,0.0,0.000000,153.000000
...,...,...,...,...,...,...,...,...,...,...,...
4333970,2.748501e+14,1.480031e+09,0.0,28.074457,-15.393759,-1.0,2.355157,170.886009,0.0,2236.013184,231.800003
4333971,2.748501e+14,1.480031e+09,0.0,28.074436,-15.393755,-1.0,2.875325,48.427730,0.0,2236.013184,234.100006
4333972,2.748501e+14,1.480031e+09,0.0,28.074453,-15.393733,-1.0,1.715123,68.198428,0.0,2236.013184,234.100006
4333973,2.748501e+14,1.480032e+09,0.0,28.074459,-15.393717,-1.0,5.744685,173.475051,0.0,2236.013184,236.800003


In [ ]:
model.save('demo_architecture.json')
model.save_weights('demo_weights.h5')

In [ ]:
# Print the model architecture
print("Model Architecture:")
print(model.summary())

Model Architecture:
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_2 (Conv1D)           (None, 8, 64)             256       
                                                                 
 max_pooling1d_2 (MaxPoolin  (None, 4, 64)             0         
 g1D)                                                            
                                                                 
 flatten_2 (Flatten)         (None, 256)               0         
                                                                 
 dense_2 (Dense)             (None, 2)                 514       
                                                                 
Total params: 770 (3.01 KB)
Trainable params: 770 (3.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None
